# Wavelet denoising and look-ahead bias in stock forecasting

Analysis notebook for the manuscript submitted to *PeerJ Computer Science*. Author: Anggit Suryopratomo (Universitas Ma'soem).

Run all cells in order (Runtime → Run all). Data are downloaded from Yahoo Finance with fixed dates (4 January 2021 to 30 October 2026).

1. Setup and data for case stocks
2. Data-quality screening of case stocks
3. Wavelet decomposition and denoising
4. Schemes A/B/C, tests and walk-forward evaluation (Tables 2–3)
5. Robustness experiment (Figures 1–2)
6. Exact test statistics (Tables 2–3)
7. LQ45 cross-section (Table 1, Figure 3, Tables S1–S2)
8. LSTM experiment (Table 4)

## 1. Setup and data for case stocks

In [ ]:
!pip install -q yfinance PyWavelets
import yfinance as yf
import numpy as np, pandas as pd, pywt
import matplotlib.pyplot as plt

saham = ["EMTK.JK", "ARTO.JK", "BBCA.JK", "MTDL.JK", "BUKA.JK", "GOTO.JK"]
for kode in saham:
    df = yf.download(kode, start="2021-01-01", end="2026-10-31", progress=False, auto_adjust=False)
    df.columns = df.columns.get_level_values(0)
    df.to_csv(f"{kode.replace('.JK','')}.csv")
    print(kode, len(df), "rows,", df.index.min().date(), "to", df.index.max().date())

## 2. Data-quality screening of case stocks

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for ax, kode in zip(axes.flat, ["EMTK", "ARTO", "BBCA", "MTDL", "BUKA", "GOTO"]):
    df = pd.read_csv(f"{kode}.csv", index_col=0, parse_dates=True)
    c = df["Close"]
    datar  = (c.diff() == 0).mean() * 100
    vol0   = (df["Volume"] == 0).sum()
    rp50   = (c <= 50).sum()
    lonjak = (c.pct_change().abs() > 0.35).sum()
    print(f"{kode}: unchanged {datar:.1f}% | zero volume: {vol0} | <=IDR50: {rp50} | jumps >35%: {lonjak}")
    ax.plot(c, linewidth=0.8); ax.set_title(kode)
plt.tight_layout(); plt.show()

## 3. Wavelet decomposition and denoising (illustration, ARTO)

In [ ]:
df = pd.read_csv("ARTO.csv", index_col=0, parse_dates=True)
x = np.log(df["Close"].values)
koef = pywt.wavedec(x, "db4", level=4)
komponen = []
for i in range(len(koef)):
    k = [c if j == i else np.zeros_like(c) for j, c in enumerate(koef)]
    komponen.append(pywt.waverec(k, "db4")[:len(x)])
print("Reconstruction check (should be ~0):", np.abs(sum(komponen) - x).max())

def denoise(sinyal, wavelet="db4", level=4):
    koef = pywt.wavedec(sinyal, wavelet, level=level)
    sigma = np.median(np.abs(koef[-1])) / 0.6745        # noise estimate from finest details
    T = sigma * np.sqrt(2 * np.log(len(sinyal)))         # universal threshold
    koef = [koef[0]] + [pywt.threshold(c, T, mode="soft") for c in koef[1:]]
    return pywt.waverec(koef, wavelet)[:len(sinyal)]

x_bersih = denoise(x)
print("Std of removed noise:", np.std(x - x_bersih).round(4))

## 4. Schemes A/B/C, statistical tests and walk-forward evaluation

In [ ]:
from sklearn.linear_model import LinearRegression

def denoise_kausal(x, jendela=256):
    hasil = np.full(len(x), np.nan)
    for t in range(jendela - 1, len(x)):
        hasil[t] = denoise(x[t - jendela + 1 : t + 1])[-1]   # hanya pakai data s.d. hari t
    return hasil

from scipy.stats import binomtest, norm

def bangun(x, sinyal, t_awal, lag=5):
    r_asli, r_fitur = np.diff(x), np.diff(sinyal)
    X = np.array([r_fitur[t - lag : t] for t in range(t_awal, len(r_asli))])
    y = r_asli[t_awal:]
    return X, y

def mcnemar(benar1, benar2):                 # apakah dua skema berbeda akurasinya?
    b = int(( benar1 & ~benar2).sum())       # skema 1 benar, skema 2 salah
    c = int((~benar1 &  benar2).sum())       # sebaliknya
    return binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0

def diebold_mariano(e1, e2):                 # apakah dua skema berbeda galatnya (kuadrat)?
    d = e1**2 - e2**2
    dm = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
    return 2 * (1 - norm.cdf(abs(dm)))

In [ ]:
def walk_forward(x, sig, t_awal, latih_awal=250, langkah=21):
    X, y = bangun(x, sig, t_awal)
    r = np.diff(x)[t_awal:]                  # return asli, sejajar dengan y
    pred, akt, drift, kemarin = [], [], [], []
    for mulai in range(latih_awal, len(y), langkah):
        selesai = min(mulai + langkah, len(y))
        model = LinearRegression().fit(X[:mulai], y[:mulai])      # latih: semua data SEBELUM bulan uji
        pred.append(model.predict(X[mulai:selesai]))
        akt.append(y[mulai:selesai])
        drift.append(np.full(selesai - mulai, np.sign(y[:mulai].mean())))
        kemarin.append(np.diff(x)[t_awal - 1 :][mulai:selesai])
    return [np.concatenate(v) for v in (pred, akt, drift, kemarin)]

tabel_wf, uji_wf = {}, {}
for kode in ["EMTK", "ARTO", "BUKA", "BBCA"]:
    x = np.log(pd.read_csv(f"{kode}.csv", index_col=0, parse_dates=True)["Close"].values)
    sinyal = {"A": x, "B": denoise(x), "C": denoise_kausal(x)}
    t_awal = 256 + 5
    pred, galat, benar = {}, {}, {}
    for s, sig in sinyal.items():
        p, akt, drift, kemarin = walk_forward(x, sig, t_awal)
        pred[s], galat[s] = p, p - akt
    m = akt != 0
    for s in pred:
        benar[s] = np.sign(pred[s][m]) == np.sign(akt[m])
    benar["Naif: ikut tren data latih"] = drift[m] == np.sign(akt[m])
    mk = kemarin[m] != 0
    benar["Naif: arah sama dengan kemarin"] = (np.sign(kemarin[m]) == np.sign(akt[m]))[mk]

    kol = {}
    for s, bnr in benar.items():
        k = int(bnr.sum()); N = len(bnr)
        kol[s] = f"{k/N*100:.1f} (p={binomtest(k, N, 0.5).pvalue:.3f})"
    kol["% hari turun di periode uji"] = f"{(akt[m] < 0).mean()*100:.1f}"
    tabel_wf[kode] = kol
    uji_wf[kode] = {
        "McNemar B vs C (arah)": round(mcnemar(benar["B"], benar["C"]), 4),
        "McNemar C vs A (arah)": round(mcnemar(benar["C"], benar["A"]), 4),
        "Diebold-Mariano B vs C (galat)": round(diebold_mariano(galat["B"], galat["C"]), 4),
        "Diebold-Mariano C vs A (galat)": round(diebold_mariano(galat["C"], galat["A"]), 4),
        "Jumlah hari uji": int(m.sum())}

tw, uw = pd.DataFrame(tabel_wf), pd.DataFrame(uji_wf)
print("WALK-FORWARD — AKURASI ARAH (%) dan p-value terhadap 50%\n", tw, "\n")
print("WALK-FORWARD — UJI ANTAR-SKEMA (p < 0,05 = berbeda nyata)\n", uw)
tw.to_csv("wf_akurasi.csv"); uw.to_csv("wf_uji.csv")

## 5. Robustness experiment (case stocks)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestRegressor

def denoise_kausal2(x, wavelet, level, jendela):
    hasil = np.full(len(x), np.nan)
    for t in range(jendela - 1, len(x)):
        hasil[t] = denoise(x[t - jendela + 1 : t + 1], wavelet, level)[-1]
    return hasil

def walk_forward2(x, sig, t_awal, buat_model, latih_awal=250, langkah=21):
    X, y = bangun(x, sig, t_awal)
    pred, akt = [], []
    for mulai in range(latih_awal, len(y), langkah):
        selesai = min(mulai + langkah, len(y))
        model = buat_model().fit(X[:mulai], y[:mulai])
        pred.append(model.predict(X[mulai:selesai])); akt.append(y[mulai:selesai])
    return np.concatenate(pred), np.concatenate(akt)

LR = lambda: LinearRegression()
RF = lambda: RandomForestRegressor(n_estimators=200, max_depth=5, random_state=0, n_jobs=-1)

# Satu faktor diubah setiap kali; sisanya tetap seperti konfigurasi utama
konfigurasi = [
    ("Utama (db4, L4, w256, LR)", "db4",  4, 256, LR),
    ("Wavelet haar",              "haar", 4, 256, LR),
    ("Wavelet sym8",              "sym8", 4, 256, LR),
    ("Level 3",                   "db4",  3, 256, LR),
    ("Level 5",                   "db4",  5, 256, LR),
    ("Jendela 128",               "db4",  4, 128, LR),
    ("Jendela 512",               "db4",  4, 512, LR),
    ("Model Random Forest",       "db4",  4, 256, RF),
]

def bintang(p): return "**" if p < 0.01 else ("*" if p < 0.05 else "")

data = {k: np.log(pd.read_csv(f"{k}.csv", index_col=0, parse_dates=True)["Close"].values)
        for k in ["EMTK", "ARTO", "BUKA", "BBCA"]}
baris = []
for nama, wv, lv, jd, mdl in konfigurasi:
    print("Menjalankan:", nama)
    for kode, x in data.items():
        t_awal = jd + 5
        sinyal = {"A": x, "B": denoise(x, wv, lv), "C": denoise_kausal2(x, wv, lv, jd)}
        benar = {}
        for s, sig in sinyal.items():
            p, akt = walk_forward2(x, sig, t_awal, mdl)
            m = akt != 0
            benar[s] = np.sign(p[m]) == np.sign(akt[m])
        acc = {s: b.mean() * 100 for s, b in benar.items()}
        p_bc, p_ca = mcnemar(benar["B"], benar["C"]), mcnemar(benar["C"], benar["A"])
        baris.append({"Konfigurasi": nama, "Saham": kode,
                      "Akurasi A": round(acc["A"], 1), "Akurasi B": round(acc["B"], 1), "Akurasi C": round(acc["C"], 1),
                      "B-C": round(acc["B"] - acc["C"], 1), "p B vs C": round(p_bc, 4),
                      "C-A": round(acc["C"] - acc["A"], 1), "p C vs A": round(p_ca, 4)})

rob = pd.DataFrame(baris)
rob.to_csv("uji_ketahanan.csv", index=False)
urut = [k[0] for k in konfigurasi]
fmt = lambda d, p: d.map(lambda v: f"{v:+.1f}") + rob[p].map(bintang)
t1 = rob.assign(v=fmt(rob["B-C"], "p B vs C")).pivot(index="Konfigurasi", columns="Saham", values="v").loc[urut]
t2 = rob.assign(v=fmt(rob["C-A"], "p C vs A")).pivot(index="Konfigurasi", columns="Saham", values="v").loc[urut]
print("\nTABEL 1. Penggelembungan akurasi B − C (poin %)   * p<0,05  ** p<0,01")
print(t1)
print("\nTABEL 2. Manfaat denoising yang benar C − A (poin %)   * p<0,05  ** p<0,01")
print(t2)
print(f"\nRingkasan: B vs C signifikan di {(rob['p B vs C'] < 0.05).sum()} dari {len(rob)} kombinasi; "
      f"C vs A signifikan di {(rob['p C vs A'] < 0.05).sum()} dari {len(rob)} kombinasi.")

## 6. Exact test statistics (case stocks)

In [ ]:
baris_stat = []
for kode in ["EMTK", "ARTO", "BUKA", "BBCA"]:
    x = np.log(pd.read_csv(f"{kode}.csv", index_col=0, parse_dates=True)["Close"].values)
    sinyal = {"A": x, "B": denoise(x), "C": denoise_kausal(x)}
    pred, galat, benar = {}, {}, {}
    for s, sig in sinyal.items():
        p, akt, drift, kemarin = walk_forward(x, sig, 256 + 5)
        pred[s], galat[s] = p, p - akt
    m = akt != 0
    for s in pred:
        benar[s] = np.sign(pred[s][m]) == np.sign(akt[m])
    benar["Naif tren"] = drift[m] == np.sign(akt[m])
    benar["Naif kemarin"] = (np.sign(kemarin[m]) == np.sign(akt[m]))[kemarin[m] != 0]
    for s, bnr in benar.items():
        k, N = int(bnr.sum()), len(bnr)
        baris_stat.append([kode, f"Akurasi {s}", f"{k}/{N} = {k/N*100:.2f}%", f"{binomtest(k, N, 0.5).pvalue:.2e}"])
    for s1, s2 in [("B", "C"), ("C", "A")]:
        b = int(( benar[s1] & ~benar[s2]).sum()); c = int((~benar[s1] & benar[s2]).sum())
        baris_stat.append([kode, f"McNemar {s1} vs {s2}", f"b={b}, c={c}", f"{binomtest(b, b + c, 0.5).pvalue:.2e}"])
        d = galat[s1]**2 - galat[s2]**2
        dm = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
        baris_stat.append([kode, f"Diebold-Mariano {s1} vs {s2}", f"DM={dm:.3f}", f"{2 * norm.sf(abs(dm)):.2e}"])

statistik = pd.DataFrame(baris_stat, columns=["Saham", "Uji", "Statistik", "p (eksak)"])
print(statistik.to_string(index=False))
statistik.to_csv("statistik_lengkap.csv", index=False)

## 7. LQ45 cross-section

In [ ]:
import os
lq45 = ["AADI","ADMR","ADRO","AKRA","AMMN","AMRT","ANTM","ASII","BBCA","BBNI","BBRI","BBTN","BMRI","BRPT","BUMI",
        "CPIN","CUAN","DEWA","EMTK","ESSA","EXCL","GOTO","HRTA","ICBP","INCO","INDF","INDY","INKP","ISAT","ITMG",
        "JPFA","KLBF","MAPI","MBMA","MDKA","MEDC","NCKL","PGAS","PGEO","PTBA","SMGR","TLKM","UNTR","UNVR","WIFI"]
kandidat = sorted(set(lq45 + ["ARTO", "BUKA"]))
os.makedirs("data_lq45", exist_ok=True)
seleksi = []
for kode in kandidat:
    try:
        df = yf.download(f"{kode}.JK", start="2021-01-01", end="2026-10-31", progress=False, auto_adjust=False)
        df.columns = df.columns.get_level_values(0)
        df = df.dropna(subset=["Close"]); df = df[df["Close"] > 0]
    except Exception as e:
        print(kode, "download failed:", e); continue
    if len(df) == 0:
        print(kode, "no data"); continue
    df.to_csv(f"data_lq45/{kode}.csv")
    c = df["Close"]
    info = {"Saham": kode, "Hari": len(df), "Mulai": df.index.min().date(),
            "Datar (%)": round((c.diff() == 0).mean() * 100, 1), "Volume 0": int((df["Volume"] == 0).sum()),
            "<=Rp50": int((c <= 50).sum()), "Lonjakan >35%": int((c.pct_change().abs() > 0.35).sum())}
    alasan = []
    if info["Hari"] < 750: alasan.append("data < 750 hari")
    if info["Datar (%)"] > 15: alasan.append("harga datar > 15%")
    if info["<=Rp50"] > 0: alasan.append("pernah di Rp50")
    if info["Lonjakan >35%"] > 0: alasan.append("lonjakan > 35%")
    info["Keputusan"] = "Masuk" if not alasan else "Keluar: " + ", ".join(alasan)
    seleksi.append(info)

seleksi = pd.DataFrame(seleksi)
seleksi.to_csv("seleksi_sampel.csv", index=False)
sampel = seleksi.loc[seleksi["Keputusan"] == "Masuk", "Saham"].tolist()
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 100)
print(seleksi.to_string(index=False))
print(f"\nFinal sample: {len(sampel)} stocks ->", sampel)

In [ ]:
from scipy.stats import wilcoxon
hasil = []
for kode in sampel:
    x = np.log(pd.read_csv(f"data_lq45/{kode}.csv", index_col=0, parse_dates=True)["Close"].values)
    sinyal = {"A": x, "B": denoise(x), "C": denoise_kausal(x)}
    pred, benar = {}, {}
    for s, sig in sinyal.items():
        p, akt, drift, kemarin = walk_forward(x, sig, 256 + 5)
        pred[s] = p
    m = akt != 0
    for s in pred:
        benar[s] = np.sign(pred[s][m]) == np.sign(akt[m])
    r = {"Saham": kode, "Hari uji": int(m.sum())}
    for s in "ABC":
        r[f"Akurasi {s}"] = round(benar[s].mean() * 100, 2)
    r["Naif tren"] = round((drift[m] == np.sign(akt[m])).mean() * 100, 2)
    for s1, s2 in [("B", "C"), ("C", "A")]:
        b = int(( benar[s1] & ~benar[s2]).sum()); c = int((~benar[s1] & benar[s2]).sum())
        r[f"{s1}-{s2}"] = round(r[f"Akurasi {s1}"] - r[f"Akurasi {s2}"], 2)
        r[f"b {s1}{s2}"], r[f"c {s1}{s2}"] = b, c
        r[f"p {s1}{s2}"] = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    hasil.append(r)
    print(f"{kode}: A={r['Akurasi A']:.1f}  B={r['Akurasi B']:.1f}  C={r['Akurasi C']:.1f}  B-C={r['B-C']:+.1f}  C-A={r['C-A']:+.1f}")

hasil = pd.DataFrame(hasil)
hasil.to_csv("hasil_lq45.csv", index=False)
n = len(hasil)
print(f"\nRINGKASAN {n} SAHAM")
for kol, uji, lab in [("B-C", "p BC", "Inflasi B − C"), ("C-A", "p CA", "Manfaat C − A")]:
    d, pk = hasil[kol], hasil[uji]
    print(f"{lab}: rata-rata {d.mean():+.2f} poin (median {d.median():+.2f}; rentang {d.min():+.1f} s.d. {d.max():+.1f}); "
          f"positif di {(d > 0).sum()}/{n} saham; McNemar p<0,05 di {(pk < 0.05).sum()}/{n} saham; Wilcoxon p = {wilcoxon(d).pvalue:.2e}")
print(f"Akurasi rata-rata: A {hasil['Akurasi A'].mean():.2f} | B {hasil['Akurasi B'].mean():.2f} | C {hasil['Akurasi C'].mean():.2f} | naif tren {hasil['Naif tren'].mean():.2f}")
print(f"C lebih akurat dari naif tren di {(hasil['Akurasi C'] > hasil['Naif tren']).sum()}/{n} saham")

## 8. LSTM experiment (case stocks; about 15–20 minutes on CPU)

In [ ]:
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

def buat_lstm(lag):
    model = tf.keras.Sequential([tf.keras.Input((lag, 1)), tf.keras.layers.LSTM(16), tf.keras.layers.Dense(1)])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

def walk_forward_lstm(x, sig, t_awal, lag=20, latih_awal=250, langkah=63, epoch=20):
    X, y = bangun(x, sig, t_awal, lag=lag)
    pred, akt = [], []
    for mulai in range(latih_awal, len(y), langkah):
        selesai = min(mulai + langkah, len(y))
        mu, sd, ys = X[:mulai].mean(), X[:mulai].std(), y[:mulai].std()   # skala hanya dari data latih
        tf.keras.utils.set_random_seed(0)
        model = buat_lstm(lag)
        model.fit(((X[:mulai] - mu) / sd)[..., None], y[:mulai] / ys, epochs=epoch, batch_size=32, verbose=0)
        pred.append(model.predict(((X[mulai:selesai] - mu) / sd)[..., None], verbose=0).ravel() * ys)
        akt.append(y[mulai:selesai])
    return np.concatenate(pred), np.concatenate(akt)

hasil_lstm = []
for kode in ["EMTK", "ARTO", "BUKA", "BBCA"]:
    x = np.log(pd.read_csv(f"{kode}.csv", index_col=0, parse_dates=True)["Close"].values)
    sinyal = {"A": x, "B": denoise(x), "C": denoise_kausal(x)}
    benar = {}
    for s, sig in sinyal.items():
        p, akt = walk_forward_lstm(x, sig, 256 + 20)
        m = akt != 0
        benar[s] = np.sign(p[m]) == np.sign(akt[m])
    r = {"Saham": kode, "Hari uji": int(m.sum())}
    for s in "ABC":
        r[f"Akurasi {s}"] = round(benar[s].mean() * 100, 1)
    for s1, s2 in [("B", "C"), ("C", "A")]:
        b = int(( benar[s1] & ~benar[s2]).sum()); c = int((~benar[s1] & benar[s2]).sum())
        r[f"{s1}-{s2}"] = round((benar[s1].mean() - benar[s2].mean()) * 100, 1)
        r[f"b/c {s1}{s2}"] = f"{b}/{c}"
        r[f"p {s1}{s2}"] = f"{binomtest(b, b + c, 0.5).pvalue:.2e}" if b + c > 0 else "1"
    hasil_lstm.append(r)
    print(r)

hasil_lstm = pd.DataFrame(hasil_lstm)
hasil_lstm.to_csv("hasil_lstm.csv", index=False)
print("\n", hasil_lstm.to_string(index=False))